# Сверка ЧОД и Фин.Рез (Excel vs Techlead month_report)

Этот ноутбук сравнивает значения `ЧОД`, `Фин.Рез` и `Количество терминалов` между тремя Excel-отчетами и витриной техлида `sandbox_ai.stg__chesnov_aef__sa_acquiring_month_report`.

Что проверяется:
- месячные агрегаты `Excel vs Techlead`;
- дельты по клиентам, терминалам, ЧОД и Финрез;
- интерпретация причин расхождений.

In [ ]:
import re

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
# Параметры Excel (каждый файл соответствует своему месяцу)
excel_sources = [
    {
        'report_month': '2026-01-01',
        'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
        'header': 1,
    },
    {
        'report_month': '2026-02-01',
        'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
        'header': 0,
    },
    {
        'report_month': '2026-03-01',
        'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
        'header': 0,
    },
]

# Основные названия и алиасы колонок
excel_col_inn = 'ИНН'
excel_col_chod = 'ЧОД'
excel_col_finrez = 'Фин.Рез'
excel_col_term_cnt = 'Количество терминалов'

tech_table = 'sandbox_ai.stg__chesnov_aef__sa_acquiring_month_report'
tech_col_month = 'trx_month'
tech_col_inn = 'inn'
tech_col_chod = 'chod'
tech_col_finrez = 'finrez'
tech_col_term_cnt = 'cnt_pos_term'
tech_col_sum_tax_amt = 'sum_tax_amt'
tech_col_fix_comiss = 'fix_comiss'
tech_col_sum_tax_irf = 'sum_tax_irf'

excel_inn_aliases = ['ИНН', 'INN']
excel_chod_aliases = ['ЧОД', 'CHOD']
excel_finrez_aliases = ['Фин.Рез', 'Фин. Рез.', 'ФИН.РЕЗ', 'ФИНРЕЗ', 'ФИН РЕЗ', 'ФИН_РЕЗ', 'FIN_RESULT']
excel_term_cnt_aliases = ['Количество терминалов', 'Кол-во терминалов', 'Терминалы', 'term_cnt', 'TERM_CNT']

# Компоненты формул в Excel (ищем по алиасам; если колонки нет, оставляем NaN)
excel_component_aliases = {
    'excel_aur': ['АУР', 'AUR', 'aur'],
    'excel_amortization': ['Амортизация', 'amortization'],
    'excel_commission_total': ['Общая комиссия', 'Комиссия итого', 'commission_total'],
    'excel_commission_from_ops': ['Комиссия %', 'Комиссия с оборота', 'commission_from_ops', 'sum_tax_amt'],
    'excel_commission_monthly': ['Комиссия руб/мес', 'Фикс комиссия', 'commission_monthly', 'fix_comiss'],
    'excel_int_component': ['Инт компонент', 'Интерчейндж', 'int_component', 'sum_tax_irf'],
}

In [ ]:
def normalize_inn(value):
    if pd.isna(value):
        return None
    s = re.sub(r'\D+', '', str(value))
    return s if s else None


def normalize_colname(value):
    s = str(value).strip().lower().replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s)
    s = s.replace('.', '').replace(',', '').replace(';', '').replace(':', '')
    s = s.replace('-', '').replace('_', '')
    return s


def pick_column(columns, aliases, col_label, file_path):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]

    available = ', '.join([str(c) for c in columns])
    raise ValueError(
        f"В файле {file_path} не найдена колонка '{col_label}'. Доступные колонки: {available}"
    )


def pick_optional_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    return None


def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s:
        return np.nan

    s = s.replace('\xa0', '').replace(' ', '')
    if ',' in s and '.' in s:
        # Если есть оба разделителя, считаем, что точка - тысячные, запятая - десятичная
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')

    s = re.sub(r'[^0-9\.-]', '', s)
    if s in {'', '-', '.', '-.'}:
        return np.nan

    try:
        return float(s)
    except Exception:
        return np.nan


def month_start(value):
    dt = pd.to_datetime(value, errors='coerce')
    if pd.isna(dt):
        return pd.NaT
    return pd.Timestamp(dt.year, dt.month, 1)

In [ ]:
# Загрузка и нормализация Excel
excel_frames = []
for src in excel_sources:
    report_month = month_start(src['report_month'])
    path = src['path']
    header = src.get('header', 0)

    df = pd.read_excel(path, header=header)

    col_inn = pick_column(df.columns, excel_inn_aliases + [excel_col_inn], 'ИНН', path)
    col_chod = pick_column(df.columns, excel_chod_aliases + [excel_col_chod], 'ЧОД', path)
    col_finrez = pick_column(df.columns, excel_finrez_aliases + [excel_col_finrez], 'Фин.Рез', path)
    col_term = pick_column(df.columns, excel_term_cnt_aliases + [excel_col_term_cnt], 'Количество терминалов', path)

    print(f"Файл: {path}")
    print(f"  header={header}")
    print(f"  ИНН -> {col_inn}")
    print(f"  ЧОД -> {col_chod}")
    print(f"  Фин.Рез -> {col_finrez}")
    print(f"  Терминалы -> {col_term}")

    cur = pd.DataFrame({
        'report_month': report_month,
        'inn': df[col_inn].apply(normalize_inn),
        'chod': df[col_chod].apply(parse_numeric),
        'fin_result': df[col_finrez].apply(parse_numeric),
        'term_cnt': df[col_term].apply(parse_numeric),
    })

    # Подтягиваем опциональные компоненты формулы из Excel
    component_hits = {}
    for target_col, aliases in excel_component_aliases.items():
        src_col = pick_optional_column(df.columns, aliases)
        component_hits[target_col] = src_col
        cur[target_col] = df[src_col].apply(parse_numeric) if src_col is not None else np.nan

    print('  Компоненты формулы:')
    for target_col, src_col in component_hits.items():
        print(f"    {target_col} <- {src_col if src_col is not None else '[не найдено]'}")

    cur['source_file'] = path
    excel_frames.append(cur)

excel_raw = pd.concat(excel_frames, ignore_index=True)
excel_raw = excel_raw[excel_raw['report_month'].notna()]

excel_qc = (
    excel_raw.groupby('report_month', as_index=False)
    .agg(
        rows=('inn', 'size'),
        inn_cnt=('inn', 'nunique'),
        chod_sum=('chod', 'sum'),
        finrez_sum=('fin_result', 'sum'),
        finrez_pos_sum=('fin_result', lambda s: s[s > 0].sum()),
        finrez_neg_sum=('fin_result', lambda s: s[s < 0].sum()),
        term_cnt_sum=('term_cnt', 'sum'),
    )
    .sort_values('report_month')
)

display(excel_qc)

In [ ]:
# Агрегация Excel до уровня месяц + ИНН
excel_by_inn = (
    excel_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        excel_chod=('chod', 'sum'),
        excel_finrez=('fin_result', 'sum'),
        excel_term_cnt=('term_cnt', 'sum'),
    )
)

excel_by_inn['excel_finrez_pos'] = np.where(excel_by_inn['excel_finrez'] > 0, excel_by_inn['excel_finrez'], 0.0)
excel_by_inn['excel_finrez_neg'] = np.where(excel_by_inn['excel_finrez'] < 0, excel_by_inn['excel_finrez'], 0.0)

excel_by_month = (
    excel_by_inn.groupby('report_month', as_index=False)
    .agg(
        excel_rows=('inn', 'size'),
        excel_inn_cnt=('inn', 'nunique'),
        excel_chod_sum=('excel_chod', 'sum'),
        excel_finrez_sum=('excel_finrez', 'sum'),
        excel_finrez_pos_sum=('excel_finrez_pos', 'sum'),
        excel_finrez_neg_sum=('excel_finrez_neg', 'sum'),
        excel_term_cnt_sum=('excel_term_cnt', 'sum'),
    )
)

# Явный показатель количества клиентов
excel_by_month['excel_client_cnt'] = excel_by_month['excel_inn_cnt']

display(excel_by_month.sort_values('report_month'))

In [ ]:
# Подготовка завершена: далее идет загрузка витрины техлида
print('Excel подготовлен. Следующий шаг: загрузка techlead month_report.')

In [ ]:
# Загрузка данных техлида из month_report (Impala)
month_min = min([month_start(x['report_month']) for x in excel_sources]).strftime('%Y-%m-%d')
month_max = max([month_start(x['report_month']) for x in excel_sources]).strftime('%Y-%m-%d')

sql_tech = f"""
select
    cast({tech_col_month} as date) as report_month,
    cast({tech_col_inn} as string) as inn,
    cast({tech_col_chod} as double) as chod,
    cast({tech_col_finrez} as double) as finrez,
    cast({tech_col_term_cnt} as double) as term_cnt
from {tech_table}
where cast({tech_col_month} as date) between cast('{month_min}' as date) and cast('{month_max}' as date)
"""

print('tech source =', tech_table)
print('month range =', month_min, '->', month_max)

# Подключение к Impala (как в твоих рабочих тетрадках)
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    tech_raw = imp.fetch(sql_tech)

tech_raw.columns = [c.lower() for c in tech_raw.columns]
tech_raw['report_month'] = pd.to_datetime(tech_raw['report_month'], errors='coerce').dt.to_period('M').dt.to_timestamp()
tech_raw['inn'] = tech_raw['inn'].apply(normalize_inn)
tech_raw['chod'] = tech_raw['chod'].apply(parse_numeric)
tech_raw['fin_result'] = tech_raw['finrez'].apply(parse_numeric)
tech_raw['term_cnt'] = tech_raw['term_cnt'].apply(parse_numeric)

tech_qc = (
    tech_raw.groupby('report_month', as_index=False)
    .agg(
        rows=('inn', 'size'),
        inn_cnt=('inn', 'nunique'),
        chod_sum=('chod', 'sum'),
        finrez_sum=('fin_result', 'sum'),
        term_cnt_sum=('term_cnt', 'sum'),
    )
    .sort_values('report_month')
)

display(tech_qc)

tech_by_inn = (
    tech_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        tech_chod=('chod', 'sum'),
        tech_finrez=('fin_result', 'sum'),
        tech_term_cnt=('term_cnt', 'sum'),
    )
)

tech_by_month = (
    tech_by_inn.groupby('report_month', as_index=False)
    .agg(
        tech_rows=('inn', 'size'),
        tech_inn_cnt=('inn', 'nunique'),
        tech_chod_sum=('tech_chod', 'sum'),
        tech_finrez_sum=('tech_finrez', 'sum'),
        tech_term_cnt_sum=('tech_term_cnt', 'sum'),
    )
)

tech_by_month['tech_client_cnt'] = tech_by_month['tech_inn_cnt']

display(tech_by_month.sort_values('report_month'))

In [ ]:
# Основная таблица: месячная сверка Excel vs Techlead
summary_excel_tech = (
    excel_by_month.merge(tech_by_month, on='report_month', how='outer')
    .sort_values('report_month')
    .reset_index(drop=True)
)

for col in [
    'excel_client_cnt', 'tech_client_cnt',
    'excel_term_cnt_sum', 'tech_term_cnt_sum',
    'excel_chod_sum', 'tech_chod_sum',
    'excel_finrez_sum', 'tech_finrez_sum'
]:
    if col in summary_excel_tech.columns:
        summary_excel_tech[col] = summary_excel_tech[col].fillna(0.0)

summary_excel_tech['diff_client_cnt_abs'] = summary_excel_tech['tech_client_cnt'] - summary_excel_tech['excel_client_cnt']
summary_excel_tech['diff_term_cnt_abs'] = summary_excel_tech['tech_term_cnt_sum'] - summary_excel_tech['excel_term_cnt_sum']
summary_excel_tech['diff_chod_abs'] = summary_excel_tech['tech_chod_sum'] - summary_excel_tech['excel_chod_sum']
summary_excel_tech['diff_finrez_abs'] = summary_excel_tech['tech_finrez_sum'] - summary_excel_tech['excel_finrez_sum']

summary_excel_tech['diff_client_cnt_pct'] = np.where(
    summary_excel_tech['excel_client_cnt'] != 0,
    summary_excel_tech['diff_client_cnt_abs'] / summary_excel_tech['excel_client_cnt'] * 100,
    np.nan,
)
summary_excel_tech['diff_term_cnt_pct'] = np.where(
    summary_excel_tech['excel_term_cnt_sum'] != 0,
    summary_excel_tech['diff_term_cnt_abs'] / summary_excel_tech['excel_term_cnt_sum'] * 100,
    np.nan,
)
summary_excel_tech['diff_chod_pct'] = np.where(
    summary_excel_tech['excel_chod_sum'] != 0,
    summary_excel_tech['diff_chod_abs'] / summary_excel_tech['excel_chod_sum'] * 100,
    np.nan,
)
summary_excel_tech['diff_finrez_pct'] = np.where(
    summary_excel_tech['excel_finrez_sum'] != 0,
    summary_excel_tech['diff_finrez_abs'] / summary_excel_tech['excel_finrez_sum'] * 100,
    np.nan,
)

display(summary_excel_tech)

# Формульный bridge для finrez: Excel fact vs Excel calc vs Techlead
bridge_df = excel_raw.copy()
for c in ['excel_aur', 'excel_amortization', 'excel_commission_total', 'excel_commission_from_ops', 'excel_commission_monthly', 'excel_int_component']:
    if c not in bridge_df.columns:
        bridge_df[c] = np.nan

# Приоритет формулы: ЧОД - АУР - Амортизация; fallback: Общая комиссия - АУР - Амортизация
bridge_df['finrez_calc_excel'] = np.where(
    bridge_df['chod'].notna(),
    bridge_df['chod'] - bridge_df['excel_aur'].fillna(0.0) - bridge_df['excel_amortization'].fillna(0.0),
    bridge_df['excel_commission_total'].fillna(0.0) - bridge_df['excel_aur'].fillna(0.0) - bridge_df['excel_amortization'].fillna(0.0),
)

bridge_month = (
    bridge_df.groupby('report_month', as_index=False)
    .agg(
        finrez_excel_fact=('fin_result', 'sum'),
        finrez_excel_calc=('finrez_calc_excel', 'sum'),
        aur_sum=('excel_aur', 'sum'),
        amortization_sum=('excel_amortization', 'sum'),
        commission_total_sum=('excel_commission_total', 'sum'),
        commission_from_ops_sum=('excel_commission_from_ops', 'sum'),
        commission_monthly_sum=('excel_commission_monthly', 'sum'),
        int_component_sum=('excel_int_component', 'sum'),
    )
    .merge(tech_by_month[['report_month', 'tech_finrez_sum', 'tech_chod_sum']], on='report_month', how='left')
)

bridge_month['delta_excel_calc_vs_fact'] = bridge_month['finrez_excel_calc'] - bridge_month['finrez_excel_fact']
bridge_month['delta_excel_fact_vs_tech'] = bridge_month['finrez_excel_fact'] - bridge_month['tech_finrez_sum']
bridge_month['delta_excel_calc_vs_tech'] = bridge_month['finrez_excel_calc'] - bridge_month['tech_finrez_sum']

display(bridge_month.sort_values('report_month'))

In [ ]:
# Интерпретация расхождений Excel vs Techlead (месячный уровень)
kpi_cols = [
    ('client_cnt', 'diff_client_cnt_pct'),
    ('term_cnt', 'diff_term_cnt_pct'),
    ('chod', 'diff_chod_pct'),
    ('finrez', 'diff_finrez_pct'),
]

kpi_drift = []
for _, row in summary_excel_tech.iterrows():
    month = row['report_month']
    for kpi, pct_col in kpi_cols:
        kpi_drift.append({
            'report_month': month,
            'kpi': kpi,
            'abs_pct_diff': abs(row.get(pct_col, np.nan)) if pd.notna(row.get(pct_col, np.nan)) else np.nan,
        })

kpi_drift_df = pd.DataFrame(kpi_drift)
kpi_drift_df = kpi_drift_df.sort_values(['report_month', 'abs_pct_diff'], ascending=[True, False])

print('KPI с максимальным относительным отклонением по месяцам:')
display(kpi_drift_df)

# Детализация month + inn по расхождениям finrez
finrez_drilldown = (
    excel_by_inn.merge(
        tech_by_inn,
        on=['report_month', 'inn'],
        how='outer',
        indicator=True,
    )
)

for col in ['excel_finrez', 'tech_finrez', 'excel_chod', 'tech_chod', 'excel_term_cnt', 'tech_term_cnt']:
    finrez_drilldown[col] = finrez_drilldown[col].fillna(0.0)

finrez_drilldown['delta_finrez'] = finrez_drilldown['excel_finrez'] - finrez_drilldown['tech_finrez']
finrez_drilldown['delta_chod'] = finrez_drilldown['excel_chod'] - finrez_drilldown['tech_chod']
finrez_drilldown['delta_term_cnt'] = finrez_drilldown['excel_term_cnt'] - finrez_drilldown['tech_term_cnt']
finrez_drilldown['abs_delta_finrez'] = finrez_drilldown['delta_finrez'].abs()

# Присоединяем компоненты excel по inn+month
excel_comp_by_inn = (
    excel_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        excel_aur=('excel_aur', 'sum'),
        excel_amortization=('excel_amortization', 'sum'),
        excel_commission_total=('excel_commission_total', 'sum'),
        excel_commission_from_ops=('excel_commission_from_ops', 'sum'),
        excel_commission_monthly=('excel_commission_monthly', 'sum'),
        excel_int_component=('excel_int_component', 'sum'),
    )
)

finrez_drilldown = finrez_drilldown.merge(excel_comp_by_inn, on=['report_month', 'inn'], how='left')

finrez_drilldown['reason_flag'] = np.select(
    [
        finrez_drilldown['_merge'] == 'left_only',
        finrez_drilldown['_merge'] == 'right_only',
        finrez_drilldown['abs_delta_finrez'] <= 0.01,
        (finrez_drilldown['delta_finrez'] * finrez_drilldown['delta_chod'] < 0),
        (finrez_drilldown['excel_aur'].fillna(0).abs() + finrez_drilldown['excel_amortization'].fillna(0).abs()) < 0.01,
        finrez_drilldown['abs_delta_finrez'] <= 10,
    ],
    [
        'missing_in_techlead',
        'missing_in_excel',
        'match',
        'sign_mismatch',
        'missing_component',
        'rounding_only',
    ],
    default='formula_component_mismatch',
)

finrez_drilldown = finrez_drilldown.drop(columns=['_merge']).sort_values(['report_month', 'abs_delta_finrez'], ascending=[True, False])

print('TOP-50 ИНН по абсолютной дельте finrez:')
display(finrez_drilldown.head(50))

# Короткий чек-лист для поиска причины
checklist = pd.DataFrame({
    'priority': [1, 2, 3, 4, 5],
    'check_item': [
        'Сравнить формулу finrez: у техлида finrez vs в Excel (включая знак и составные части).',
        'Сравнить формулу CHOD и состав комиссии (sum_tax_amt, fix_comiss, sum_tax_irf).',
        'Проверить определение term_cnt: cnt_pos_term у техлида vs логика в Excel.',
        'Проверить периметр клиентов за месяц (даты и фильтры актуальности).',
        'Проверить дубликаты по inn+month в Excel и витрине техлида.',
    ]
})

print('Приоритетный чек-лист поиска ошибки:')
display(checklist)

In [ ]:
# Итоговый root-cause summary по месяцам
root_cause_summary = (
    finrez_drilldown[finrez_drilldown['reason_flag'] != 'match']
    .groupby(['report_month', 'reason_flag'], as_index=False)
    .agg(
        inn_cnt=('inn', 'nunique'),
        rows=('inn', 'size'),
        abs_finrez_delta_sum=('abs_delta_finrez', 'sum'),
    )
)

month_total = root_cause_summary.groupby('report_month', as_index=False).agg(month_abs_delta=('abs_finrez_delta_sum', 'sum'))
root_cause_summary = root_cause_summary.merge(month_total, on='report_month', how='left')
root_cause_summary['share_pct'] = np.where(
    root_cause_summary['month_abs_delta'] != 0,
    root_cause_summary['abs_finrez_delta_sum'] / root_cause_summary['month_abs_delta'] * 100,
    np.nan,
)

root_cause_summary = root_cause_summary.sort_values(['report_month', 'abs_finrez_delta_sum'], ascending=[True, False])

print('Root-cause summary по месяцам:')
display(root_cause_summary)

priority_actions = (
    root_cause_summary.sort_values(['report_month', 'share_pct'], ascending=[True, False])
    .groupby('report_month', as_index=False)
    .first()[['report_month', 'reason_flag', 'share_pct']]
    .rename(columns={'reason_flag': 'top_reason_flag', 'share_pct': 'top_reason_share_pct'})
)

print('Приоритет проверки по месяцам (топ причина):')
display(priority_actions)

print('Расчеты завершены. Выгрузка в файлы отключена.')